# 02. Clean the comments and label sentiment

**Reads:** `data/youtube_comments.csv`
**Writes:** `data/comments_clean.csv`, `data/comments_sample.csv`, `data/comments_sentiment.csv`

**Install once** (terminal): `pip install emoji transformers torch sentencepiece`

**Part A, cleaning.** Comments mix English, Urdu script and Roman Urdu, plus emoji, links and video timestamps. We keep the original text and add cleaned columns, so nothing is lost.

**Part B, sentiment.** The model `cardiffnlp/twitter-xlm-roberta-base-sentiment` (multilingual XLM-RoBERTa) labels each comment positive, neutral or negative. It was not trained specifically on Roman Urdu, so results are indicative, not exact. Because the machine has no GPU, the model runs on a stratified sample of 500 comments per drama (spread evenly over episodes).

## Setup

In [1]:
import os, re, html, glob, warnings
import pandas as pd
import emoji

warnings.filterwarnings("ignore")
os.makedirs("data", exist_ok=True)

N_PER_DRAMA = 500       # comments per drama for the sentiment model

## Part A. Cleaning

Steps: decode HTML entities, remove links, video timestamps and @mentions, shorten stretched letters ("sooooo" becomes "soo"), turn emoji into words for the model, then drop empty comments, duplicates within a drama and one-word comments.
Roman Urdu spelling is **not** normalised, because spelling varies too much for simple rules.

In [2]:
df = pd.read_csv("data/youtube_comments.csv").drop(columns="author", errors="ignore")
df["text_raw"] = df["text"].astype(str)

URL   = re.compile(r"https?://\S+|www\.\S+")
TIME  = re.compile(r"\b\d{1,2}:\d{2}(?::\d{2})?\b")
MENTN = re.compile(r"@\S+")
SPACE = re.compile(r"\s+")
URDU  = re.compile(r"[\u0600-\u06FF]")

def clean(t):
    t = html.unescape(t)
    t = URL.sub(" ", t)
    t = TIME.sub(" ", t)
    t = MENTN.sub(" ", t)
    t = re.sub(r"(.)\1{3,}", r"\1\1", t)
    return SPACE.sub(" ", t).strip()

def emoji_to_text(t):
    return emoji.demojize(t, delimiters=(" ", " ")).replace("_", " ")

df["has_urdu_script"] = df["text_raw"].str.contains(URDU)
df["emoji_count"]     = df["text_raw"].apply(emoji.emoji_count)
df["text_clean"]      = df["text_raw"].apply(clean)
df["text_model"]      = df["text_clean"].apply(emoji_to_text).str.lower()
df["words"]           = df["text_model"].str.split().str.len()

before = len(df)
df = df[df["text_clean"].str.len() > 0]
df = df.drop_duplicates(["drama", "text_clean"])
df = df[df["words"] >= 2].reset_index(drop=True)

df.to_csv("data/comments_clean.csv", index=False)
print(f"{before} -> {len(df)} comments kept")

159233 -> 130756 comments kept


In [3]:
df["script"] = df["has_urdu_script"].map({True: "Urdu script", False: "Latin (English / Roman Urdu)"})
print(df["script"].value_counts(normalize=True).round(3))
df.groupby("drama").size().rename("comments")

script
Latin (English / Roman Urdu)    0.974
Urdu script                     0.026
Name: proportion, dtype: float64


drama
Ehd-e-Wafa        9832
Humsafar          7128
Ishq Murshid     12759
Jaan Nisar       24317
Mere Humsafar    17702
Parizaad         12666
Sinf-e-Aahan      9776
Suno Chanda      12870
Tere Bin         23706
Name: comments, dtype: int64

## Part B. Sentiment

The sample takes the same number of comments from every episode of a drama, then tops up randomly to 500. Scores are saved in chunks of 1,000 in `data/sent_parts/`, so an interrupted run resumes where it stopped.

If you change `N_PER_DRAMA`, delete the `data/sent_parts` folder first, otherwise old chunks will not match the new sample.

In [4]:
import torch
from transformers import pipeline

def sample_drama(g, n=N_PER_DRAMA):
    per_ep = max(1, n // g["episode"].nunique())
    s = g.groupby("episode", group_keys=False).apply(
        lambda x: x.sample(min(len(x), per_ep), random_state=42))
    if len(s) < n:
        rest = g.drop(s.index)
        s = pd.concat([s, rest.sample(min(len(rest), n - len(s)), random_state=42)])
    return s

sample = (df.groupby("drama", group_keys=False).apply(sample_drama).reset_index(drop=True))
sample.to_csv("data/comments_sample.csv", index=False)
print(len(sample), "comments in sample")
sample.groupby("drama").size()

4500 comments in sample


drama
Ehd-e-Wafa       500
Humsafar         500
Ishq Murshid     500
Jaan Nisar       500
Mere Humsafar    500
Parizaad         500
Sinf-e-Aahan     500
Suno Chanda      500
Tere Bin         500
dtype: int64

In [5]:
torch.set_num_threads(os.cpu_count())
clf = pipeline("sentiment-analysis", model="cardiffnlp/twitter-xlm-roberta-base-sentiment",
               device=-1, top_k=None, truncation=True, max_length=64)

os.makedirs("data/sent_parts", exist_ok=True)
CH = 1000
for i in range(0, len(sample), CH):
    f = f"data/sent_parts/part_{i // CH:03d}.csv"
    if os.path.exists(f):
        continue
    part = sample.iloc[i:i + CH]
    out = clf(part["text_clean"].tolist(), batch_size=32)
    res = pd.DataFrame([{d["label"].lower(): d["score"] for d in o} for o in out])
    res.insert(0, "comment_id", part["comment_id"].values)
    res.to_csv(f, index=False)
    print(f"done {min(i + CH, len(sample))}/{len(sample)}")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 12843.03it/s]


In [6]:
scores = pd.concat([pd.read_csv(f) for f in sorted(glob.glob("data/sent_parts/*.csv"))])
result = sample.merge(scores, on="comment_id", how="left")
result["sentiment"]  = result[["negative", "neutral", "positive"]].idxmax(axis=1)
result["sent_score"] = result["positive"] - result["negative"]
result.to_csv("data/comments_sentiment.csv", index=False)

print(result["sentiment"].value_counts(normalize=True).round(3))
result.groupby("drama")["sentiment"].value_counts(normalize=True).unstack().round(3)

sentiment
positive    0.448
negative    0.310
neutral     0.242
Name: proportion, dtype: float64


sentiment,negative,neutral,positive
drama,,,
Ehd-e-Wafa,0.216,0.326,0.458
Humsafar,0.472,0.178,0.350
Ishq Murshid,0.228,0.284,0.488
Jaan Nisar,0.370,0.174,0.456
Mere Humsafar,0.404,0.216,0.380
Parizaad,0.252,0.272,0.476
Sinf-e-Aahan,0.216,0.208,0.576
Suno Chanda,0.252,0.292,0.456
Tere Bin,0.424,0.221,0.355


**Reading the result**
- `sentiment` is the label with the highest score; `sent_score` is positive minus negative (from -1 to +1).
- With 500 comments per drama, each percentage has roughly a plus or minus 4 point margin.
- "Negative" often means anger at a villain, which can be praise for the drama.